# Suite VLG — The Verilog language

Verilog and SystemVerilog as one tree language (`mbse.Programs.Verilog.Syntax`): its kinds and categories,
organized as IEEE 1800's grammar is, the identifiers it accepts, and where each kind and feature exists in the two
families, which the standards check.

In [ ]:
from mbse.Programs.Framework import Syntax as F
from mbse.Programs.Verilog import SystemVerilog2017, SystemVerilog2023, Syntax as S, Verilog2005, VerilogStandard

L = S.LANGUAGE
V1995, V2001 = VerilogStandard(1995, "Verilog"), VerilogStandard(2001, "Verilog")
SV2005, SV2009, SV2012 = VerilogStandard(2005), VerilogStandard(2009), VerilogStandard(2012)


def name(spelling):
    return S.Identifier(spelling=spelling)


def ref(spelling):
    return S.NameExpression(name=name(spelling))


def number(spelling="1"):
    return S.IntegerLiteral(spelling=spelling)


def unit(*items):
    return S.SourceText(items=list(items))


def module(*items, **properties):
    return S.ModuleDeclaration(keyword="module", name=name("m"), items=list(items), **properties)

## VLG-01 · Every kind belongs to one category, and the grammar lists them

In [ ]:
kinds = L.kinds()
assert len(kinds) == len(S.KINDS) == 210 and list(kinds) == [k.__name__ for k in S.KINDS]
assert sorted(L.categories()) == sorted(["Name", "Item", "Port", "DataType", "Dimension", "Statement", "Connection",
                                        "TimingControl", "Expression", "Literal", "Range", "Directive", "Constraint", "Property",
                                        "Sequence", "BinsSelect"])
assert issubclass(S.IntegerLiteral, S.Literal) and issubclass(S.Literal, S.Expression)
assert all(k.NAME == f"Programs.Verilog.{k.KIND}" for k in S.KINDS)
grammar = L.grammar()
assert grammar["base"] == {"Verilog": 1995, "SystemVerilog": 2005} and len(grammar["kinds"]) == 210
assert S.sv() == {"SystemVerilog": 2005} and S.sv(2009) == {"SystemVerilog": 2009}
assert S.verilog(2001) == {"Verilog": 2001, "SystemVerilog": 2005}
assert S.Comment(text=" x").block is False and S.SourceText().items == []
print("ok")

## VLG-02 · Identifiers are simple or escaped, and system names start with `$`

In [ ]:
assert L.validate(name("x_1$")) == [] and L.validate(name("\\bus+index")) == [] and L.validate(name("")) == []
for bad in ["1x", "$x", "a b", "Größe", "\\", "\\a b"]:
    assert L.validate(name(bad)) == [f"{bad!r} is not an identifier"], bad
assert L.validate(S.Identifier(spelling=3)) == ["Identifier.spelling must be a str, got int"]
assert L.validate(S.SystemCall(name="$clog2")) == [] and L.validate(S.SystemCall(name="")) == []
for bad in ["clog2", "$", "$a b", "$é"]:
    assert L.validate(S.SystemCall(name=bad)) == [f"{bad!r} is not a system task or function name"], bad
assert L.validate(S.SystemCall(name=1)) == ["SystemCall.name must be a str, got int"]
print("ok")

## VLG-03 · Kinds and features exist by family and year: Verilog has none of SystemVerilog's own

In [ ]:
labels = [Verilog2005.STANDARD.name(), SystemVerilog2017.STANDARD.name(), SystemVerilog2023.STANDARD.name(), V1995.name()]
assert labels == ["Verilog-2005", "SystemVerilog-2017", "SystemVerilog-2023", "Verilog-1995"]
package = S.PackageDeclaration(name=name("p"))
assert Verilog2005.check(package) == ["PackageDeclaration is not Verilog"] and SV2005.check(package) == []
ansi = S.AnsiPort(direction="input", name=name("a"))
assert V1995.check(ansi) == ["AnsiPort needs Verilog-2001"] and V2001.check(ansi) == []
logic = S.IntegerVectorType(keyword="logic")
assert Verilog2005.check(logic) == ["IntegerVectorType.keyword 'logic' is not Verilog"]
assert Verilog2005.check(S.IntegerVectorType(keyword="reg", signing="signed")) == []
assert V1995.check(S.IntegerVectorType(keyword="reg", signing="signed")) == ["IntegerVectorType.signing 'signed' needs Verilog-2001"]
unique0 = S.IfStatement(qualifier="unique0", condition=ref("a"), consequence=S.NullStatement())
assert SV2005.check(unique0) == ["IfStatement.qualifier 'unique0' needs SystemVerilog-2009"] and SV2009.check(unique0) == []
final = S.ImmediateAssertion(keyword="assert", deferral="final", expression=ref("a"))
assert SV2009.check(final) == ["ImmediateAssertion.deferral 'final' needs SystemVerilog-2012"] and SV2012.check(final) == []
star = S.EventControl()
assert V1995.check(star) == ["EventControl * needs Verilog-2001"] and V2001.check(star) == []
fork = S.DisableStatement()
assert Verilog2005.check(fork) == ["DisableStatement fork is not Verilog"]
assert Verilog2005.check(S.DisableStatement(target=ref("b"))) == []
declared = S.ForStatement(initializers=[S.VariableDeclaration(type=S.IntegerAtomType(keyword="integer"), declarators=[
    S.VariableDeclarator(name=name("i"), value=number())])], body=S.NullStatement())
assert Verilog2005.check(declared) == ["ForStatement.initializers declarations is not Verilog"]
two = S.ForStatement(initializers=[S.AssignmentExpression(target=ref("i"), operator="=", value=number()),
                                   S.AssignmentExpression(target=ref("j"), operator="=", value=number())], body=S.NullStatement())
assert Verilog2005.check(two) == ["ForStatement.initializers declarations is not Verilog"] and SV2005.check(two) == []
one = S.ForStatement(initializers=[S.AssignmentExpression(target=ref("i"), operator="=", value=number())], body=S.NullStatement())
assert Verilog2005.check(one) == []
labeled = module(labeled=True)
assert Verilog2005.check(labeled) == ["ModuleDeclaration.labeled is not Verilog"]
assert V1995.check(S.BinaryExpression(left=number(), operator="**", right=number())) == [
    "BinaryExpression.operator '**' needs Verilog-2001"]
assert SV2005.check(S.BinaryExpression(left=number(), operator="<->", right=number())) == [
    "BinaryExpression.operator '<->' needs SystemVerilog-2009"]
interface_class = S.ClassDeclaration(interface=True, name=name("i"))
assert Verilog2005.check(interface_class) == ["ClassDeclaration is not Verilog", "ClassDeclaration.interface is not Verilog"]
assert SV2009.check(interface_class) == ["ClassDeclaration.interface needs SystemVerilog-2012"]
assert SV2009.check(S.ForwardTypedefDeclaration(keyword="interface class", name=name("i"))) == [
    "ForwardTypedefDeclaration.keyword 'interface class' needs SystemVerilog-2012"]
assert Verilog2005.check(S.VariableDeclaration(random="rand", declarators=[S.VariableDeclarator(name=name("r"))])) == [
    "VariableDeclaration.random is not Verilog"]
assert Verilog2005.check(S.NewExpression()) == ["NewExpression is not Verilog"]
print("ok")

## VLG-04 · A standard checks a whole tree, by path

In [ ]:
tree = unit(module(S.AlwaysConstruct(keyword="always_ff", body=S.NullStatement()),
                   S.ContinuousAssign(assignments=[S.AssignmentExpression(target=ref("a"), operator="+=", value=number())])))
assert Verilog2005.check(tree) == ["items[0].items[0]: AlwaysConstruct.keyword 'always_ff' is not Verilog",
                                   "items[0].items[1].assignments[0]: AssignmentExpression.operator '+=' is not Verilog"]
assert SystemVerilog2023.check(tree) == []
print("ok")

## VLG-05 · Classes and methods check their own shape

In [ ]:
def method(kind, **properties):
    return kind(name=name("f"), **properties)


for kind in (S.FunctionDeclaration, S.TaskDeclaration):
    assert L.validate(method(kind, virtual=True, pure=True)) == [] and L.validate(method(kind, extern=True)) == []
    assert L.validate(method(kind, pure=True)) == [f"a pure {kind.__name__} is virtual"]
    assert L.validate(method(kind, extern=True, body=[S.NullStatement()])) == [f"an extern or pure {kind.__name__} has no body"]
    assert L.validate(method(kind, virtual=True, pure=True, labeled=True)) == [f"an extern or pure {kind.__name__} has no body"]
base = S.NamedType(name=name("b"))
assert L.validate(S.ClassDeclaration(name=name("c"), base=base, arguments=[number()])) == []
assert L.validate(S.ClassDeclaration(interface=True, name=name("i"), interfaces=[S.NamedType(name=name("j"))])) == []
assert L.validate(S.ClassDeclaration(interface=True, name=name("i"), base=S.NamedType(name=name("b")))) == [
    "an interface ClassDeclaration has no virtual, base or arguments"]
assert L.validate(S.ClassDeclaration(interface=True, virtual=True, name=name("i"))) == [
    "an interface ClassDeclaration has no virtual, base or arguments"]
assert L.validate(S.ClassDeclaration(name=name("c"), arguments=[number()])) == ["a ClassDeclaration with arguments has a base"]
print("ok")

## VLG-06 · Distributions check their own shape, and where constraints and randomization exist

In [ ]:
assert L.validate(S.DistItem(value=number())) == [] and L.validate(S.DistItem(value=number(), operator=":=", weight=number())) == []
for item in (S.DistItem(value=number(), operator=":="), S.DistItem(value=number(), weight=number())):
    assert L.validate(item) == ["a DistItem has both an operator and a weight, or neither"]
assert L.validate(S.DistItem(operator=":=", weight=number())) == ["a default DistItem weighs its values with :/"]
assert L.validate(S.DistItem()) == ["a default DistItem weighs its values with :/"]
assert L.validate(S.DistItem(operator=":/", weight=number())) == []
assert SystemVerilog2017.check(S.DistItem(operator=":/", weight=number())) == ["DistItem default needs SystemVerilog-2023"]
assert SV2009.check(S.ExpressionConstraint(soft=True, expression=ref("a"))) == ["ExpressionConstraint.soft needs SystemVerilog-2012"]
assert SV2009.check(S.ConstraintPrototype(qualifier="pure", name=name("k"))) == [
    "ConstraintPrototype.qualifier 'pure' needs SystemVerilog-2012"]
assert SV2009.check(S.UniqueConstraint(set=[ref("a")])) == ["UniqueConstraint needs SystemVerilog-2012"]
assert SV2005.check(S.LocalName(name=name("x"))) == ["LocalName needs SystemVerilog-2009"]
assert SV2009.check(S.RandomizeWithExpression(call=ref("r"), restricted=True)) == [
    "RandomizeWithExpression.restricted needs SystemVerilog-2012"]
assert Verilog2005.check(S.RandCaseStatement()) == ["RandCaseStatement is not Verilog"]
print("ok")


## VLG-07 · Where assertions, their properties and clocking exist

In [ ]:
restrict = S.ConcurrentAssertion(keyword="restrict", spec=S.PropertySpec(property=ref("a")))
assert SV2005.check(restrict) == ["ConcurrentAssertion.keyword 'restrict' needs SystemVerilog-2009"] and SV2009.check(restrict) == []
assert Verilog2005.check(restrict) == ["ConcurrentAssertion is not Verilog", "ConcurrentAssertion.keyword 'restrict' is not Verilog",
                                       "spec: PropertySpec is not Verilog"]
since_2009 = [S.ImplicationProperty(antecedent=ref("a"), operator="#-#", consequent=ref("b")),
              S.BinaryProperty(left=ref("a"), operator="until", right=ref("b")), S.UnaryProperty(operator="always", operand=ref("a")),
              S.StrengthProperty(keyword="strong", sequence=ref("a")),
              S.AbortProperty(keyword="accept_on", condition=ref("a"), operand=ref("b")), S.CaseProperty(expression=ref("a")),
              S.LetDeclaration(name=name("l"), value=ref("a")), S.DefaultDisable(condition=ref("a")),
              S.ClockingDeclaration(scope="global", clock=S.EventControl()), S.AssertionPort(local=True, name=name("p")),
              S.KeywordType(keyword="untyped")]
for node in since_2009:
    problems = SV2005.check(node)
    assert problems and all("needs SystemVerilog-2009" in p for p in problems) and SV2009.check(node) == [], (node, problems)
assert SV2005.check(S.UnaryProperty(operator="not", operand=ref("a"))) == []
assert SV2005.check(S.ImplicationProperty(antecedent=ref("a"), operator="|->", consequent=ref("b"))) == []
print("ok")


## VLG-08 · Covergroups, coverpoints, bins and transitions check their own shape, and where they exist

In [ ]:
clock = S.EventControl(events=[S.EventExpression(expression=ref("clk"))])
assert L.validate(S.CovergroupDeclaration(name=name("g"), clock=clock, sample=S.SampleFunction())) == [
    "a CovergroupDeclaration has a clock or a sample, not both"]
assert L.validate(S.Coverpoint(type=S.IntegerAtomType(keyword="int"), expression=ref("a"))) == [
    "a Coverpoint with a type has a label"]
assert L.validate(S.CoverageBins(keyword="bins", name=name("b"), size=number(), initializer=S.BinsDefault())) == [
    "a CoverageBins with a size is an array"]
assert L.validate(S.CoverageBins(keyword="bins", name=name("b"), array=True, size=number(), initializer=S.BinsDefault())) == []
assert L.validate(S.TransitionStep(values=[number()], operator="*")) == [
    "a TransitionStep has both an operator and a count, or neither"]
assert L.validate(S.TransitionStep(values=[number()], operator="*", count=number())) == []
assert Verilog2005.check(S.CovergroupDeclaration(name=name("g"))) == ["CovergroupDeclaration is not Verilog"]
since_2012 = [S.BinsExpression(expression=ref("a")), S.FilteredBinsSelect(select=ref("a"), filter=ref("b")),
              S.BinsValues(values=[number()], filter=ref("item")),
              S.Coverpoint(label=name("c"), type=S.IntegerAtomType(keyword="int"), expression=ref("a"))]
for node in since_2012:
    problems = SV2009.check(node)
    assert problems and all("needs SystemVerilog-2012" in p for p in problems) and SV2012.check(node) == [], (node, problems)
assert SV2005.check(S.SampleFunction()) == ["SampleFunction needs SystemVerilog-2009"]
print("ok")


## VLG-09 · Attributes exist from Verilog-2001

In [ ]:
keep = [S.AttributeInstance(specs=[S.AttributeSpec(name=name("keep"))])]
wire = S.NetDeclaration(net_type="wire", declarators=[S.VariableDeclarator(name=name("w"))])
attributed = S.AttributedItem(attributes=keep, item=wire)
assert V1995.check(attributed) == ["AttributedItem needs Verilog-2001", "attributes[0]: AttributeInstance needs Verilog-2001",
                                   "attributes[0].specs[0]: AttributeSpec needs Verilog-2001"]
assert V2001.check(attributed) == []
for node in [S.BinaryExpression(left=ref("a"), operator="+", attributes=[S.AttributeInstance()], right=ref("b")),
             S.UnaryExpression(operator="-", attributes=[S.AttributeInstance()], operand=ref("a")),
             S.ConditionalExpression(condition=ref("a"), attributes=[S.AttributeInstance()], consequence=ref("b"),
                                     alternative=ref("c")),
             S.CallExpression(callee=ref("f"), attributes=[S.AttributeInstance()]),
             S.SystemCall(name="$f", attributes=[S.AttributeInstance()])]:
    problems = V1995.check(node)
    assert f"{type(node).__name__}.attributes needs Verilog-2001" in problems and V2001.check(node) == [], problems
print("ok")


## VLG-10 · Streaming, delays, patterns, tolerance ranges and the names and types SystemVerilog adds check their own shape, and where they exist

In [ ]:
a, b, c = ref("a"), ref("b"), ref("c")
assert L.validate(S.DelayControl(value=a, fall=b, turnoff=c)) == [] and L.validate(S.DelayControl(value=a, fall=b)) == []
assert L.validate(S.DelayControl(value=a, turnoff=c)) == ["a DelayControl with a turnoff has a fall"]
pattern = S.AssignmentPattern(count=number("2"), items=[a, b])
assert L.validate(pattern) == [] and SV2005.check(pattern) == []
assert L.validate(S.AssignmentPattern(count=number("2"), items=[S.PatternItem(key=a, value=b)])) == [
    "a repeated AssignmentPattern has positional items only"]
for item in (S.StreamItem(expression=a), S.StreamItem(expression=a, left=b),
             S.StreamItem(expression=a, left=b, operator="+:", right=c)):
    assert L.validate(item) == [], item
for item in (S.StreamItem(expression=a, left=b, operator=":"), S.StreamItem(expression=a, left=b, right=c),
             S.StreamItem(expression=a, operator=":", right=c)):
    assert L.validate(item) == ["a StreamItem's `with` has a left, and an operator and a right, or neither"], item
stream = S.StreamingConcatenation(operator="<<", slice=S.IntegerAtomType(keyword="byte"),
                                  items=[S.StreamItem(expression=a, left=b)])
assert SV2005.check(stream) == ["items[0]: StreamItem.left needs SystemVerilog-2009"] and SV2009.check(stream) == []
assert V2001.check(stream) == ["StreamingConcatenation is not Verilog", "slice: IntegerAtomType.keyword 'byte' is not Verilog",
                               "items[0]: StreamItem is not Verilog", "items[0]: StreamItem.left is not Verilog"]
tolerance = S.ValueRange(left=a, operator="+/-", right=b)
assert SystemVerilog2017.check(tolerance) == ["ValueRange.operator needs SystemVerilog-2023"]
assert SystemVerilog2023.check(tolerance) == []
assert V1995.check(S.MinTypMaxExpression(min=a, typ=b, max=c)) == [] and V1995.check(S.EmptyArgument()) == []
for node in (S.RootExpression(), S.EmptyQueue(), S.UnitName(), S.TypeReference(operand=a)):
    assert V2001.check(node) == [f"{type(node).__name__} is not Verilog"] and SV2005.check(node) == []
scoped = S.ScopedName(scope=S.UnitName(), name=name("x"))
assert L.validate(scoped) == [] and V2001.check(scoped) == ["ScopedName is not Verilog", "scope: UnitName is not Verilog"]
print("ok")
